# Emojis classification

Baseline (see `main-colab.ipynb`): a CNN trained from scratch on 72 px images reached 75.6% validation accuracy and 74.95% on the Kaggle test set. It overfitted (93% training accuracy) and its validation accuracy kept swinging between epochs, because its batch normalization layers behaved differently in training and evaluation mode.

This notebook improves on it with:
1. **Transfer learning**: a pretrained EfficientNetV2-B0, fine-tuned on the emojis, with its batch normalization layers kept in evaluation mode
2. **Higher resolution**: 128 px instead of 72 px, to keep the fine details of the 160 px and 240 px source images
3. **Better data augmentation**: white fill instead of mirrored edges, small rotations, and no horizontal flip
4. **Regularization**: label smoothing, weight decay (AdamW) and a cosine learning rate schedule
5. **5-fold stratified cross-validation**: a more reliable accuracy estimate, and the 5 fold models are averaged for the test predictions (ensemble)

## Imports
### Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

In [ ]:
# On Colab, read the data from Google Drive and save outputs there, so they survive the session
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/data")
    OUTPUT_DIR = Path("/content/drive/MyDrive/emoji_outputs")
except ImportError:
    DATA_DIR = Path("data")
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs: {gpus}")
if gpus:
    # Compute in float16 on the GPU (about 2x faster on a T4), weights stay in float32
    tf.keras.mixed_precision.set_global_policy("mixed_float16")

### Data

In [ ]:
IMG_SIZE = 128

def load_image(img_file: Path) -> np.ndarray:
    # Images mix RGBA and palette PNGs: composite onto a white background to get a uniform RGB array
    img = Image.open(img_file).convert("RGBA")
    background = Image.new("RGBA", img.size, (255, 255, 255, 255))
    img = Image.alpha_composite(background, img).convert("RGB")
    # Images come in several sizes (64 to 240 px, some non-square): pad to a white square, then resize
    side = max(img.size)
    square = Image.new("RGB", (side, side), (255, 255, 255))
    square.paste(img, ((side - img.width) // 2, (side - img.height) // 2))
    return np.array(square.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS))

def load_data(folder_path: Path):
    ids = []
    dataset = []
    for img_file in sorted(folder_path.iterdir()):
        if img_file.suffix != ".png":
            continue
        ids.append(img_file.stem)
        dataset.append(load_image(img_file))
    dataset_as_array = np.array(dataset)
    return dataset_as_array, ids

In [ ]:
X_train, X_train_ids = load_data(DATA_DIR / "train")
X_test, X_test_ids = load_data(DATA_DIR / "test")

labels = pd.read_csv(DATA_DIR / "train_labels.csv", dtype={"Id": str}).set_index("Id")
y_train = labels.loc[X_train_ids, "Label"].to_numpy()

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for ax, img, label in zip(axes.flat, X_train, y_train):
    ax.imshow(img)
    ax.set_title(label)
    ax.axis("off")
plt.show()

### Preprocessing

Pixel values stay in [0, 255] (as `uint8`, which also saves memory): EfficientNetV2 rescales its inputs itself.

In [ ]:
# Encode string labels as integers, then one-hot encode
classes = np.array(sorted(set(y_train)))
num_classes = len(classes)
y_train_int = np.searchsorted(classes, y_train)
y_train_cat = to_categorical(y_train_int, num_classes=num_classes)

print(f"Classes: {list(classes)}")
print(f"Categorical y_train shape: {y_train_cat.shape}")

### Cross-validation folds

The training set is split into 5 folds. Each fold model trains on 4 folds and is validated on the remaining one, so every training image is used for validation exactly once. The folds are stratified: each keeps the vendor proportions of the full training set, which matters for the rare vendors (Mozilla has only 397 images).

In [ ]:
N_FOLDS = 5
# For a quick experiment, train a single fold with FOLDS_TO_RUN = [0]; the final submission should use all folds
FOLDS_TO_RUN = list(range(N_FOLDS))

rng = np.random.default_rng(0)
fold_of = np.empty(len(y_train_int), dtype=int)
for c in range(num_classes):
    class_idx = rng.permutation(np.flatnonzero(y_train_int == c))
    fold_of[class_idx] = np.arange(len(class_idx)) % N_FOLDS

pd.crosstab(fold_of, y_train, rownames=["fold"], colnames=["vendor"])

## Model
### Model architecture

- **Data augmentation** (training only): random shifts (±10%), zoom (±15%) and rotation (±11°). Empty areas are filled with white, like the emoji backgrounds (the baseline mirrored the image edges instead). There is no horizontal flip: vendor styles may have a consistent light direction (e.g. highlights on the upper left), which a flip would destroy.
- **EfficientNetV2-B0** pretrained on ImageNet, without its classification layer. Its low-level features (edges, gradients, shading, textures) are what distinguishes vendor styles. With about 10,000 training images, a pretrained network overfits much less than a network trained from scratch.
- Global average pooling, dropout 30%, then a 7-neuron softmax layer.

The base is called with `training=False`: its batch normalization layers always use their stored ImageNet statistics, even while fine-tuning. This avoids the training/evaluation mismatch that made the baseline's validation accuracy unstable.

In [ ]:
def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # Augmentation layers are only active during training
    x = layers.RandomTranslation(0.1, 0.1, fill_mode="constant", fill_value=255.0)(inputs)
    x = layers.RandomZoom(0.15, fill_mode="constant", fill_value=255.0)(x)
    x = layers.RandomRotation(0.03, fill_mode="constant", fill_value=255.0)(x)

    base = EfficientNetV2B0(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    # Keep batch normalization in inference mode, even once the base is unfrozen
    x = base(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    # float32 output for numerical stability under mixed precision
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    return models.Model(inputs, outputs), base

model, base = build_model()
model.summary()

### Model training parameters

Each fold model is trained in two stages:
1. **Head only** (3 epochs): the pretrained base is frozen and only the new classification layer learns, so that its random initial weights don't send large, destructive gradients into the pretrained base.
2. **Fine-tuning** (up to 25 epochs): the whole network trains with a small learning rate (2e-4) that decreases smoothly to 0 (cosine decay), using AdamW (Adam with weight decay).

The loss uses label smoothing (0.1): targets become 0.9 / 0.017 instead of 1 / 0, which discourages overconfident predictions and reduces overfitting. Training stops when validation accuracy has not improved for 6 epochs, and the best epoch's weights are kept.

In [ ]:
HEAD_EPOCHS = 3
FINETUNE_EPOCHS = 25
BATCH_SIZE = 64

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

def train_fold(fold):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(fold)
    is_train = fold_of != fold
    X_tr, y_tr = X_train[is_train], y_train_cat[is_train]
    X_val, y_val = X_train[~is_train], y_train_cat[~is_train]

    model, base = build_model()

    # Stage 1: train only the classification head
    base.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=loss, metrics=["accuracy"])
    head_history = model.fit(
        X_tr, y_tr, validation_data=(X_val, y_val), epochs=HEAD_EPOCHS, batch_size=BATCH_SIZE
    )

    # Stage 2: fine-tune the whole network
    base.trainable = True
    steps_per_epoch = int(np.ceil(len(X_tr) / BATCH_SIZE))
    schedule = tf.keras.optimizers.schedules.CosineDecay(2e-4, decay_steps=FINETUNE_EPOCHS * steps_per_epoch)
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=schedule), loss=loss, metrics=["accuracy"])

    weights_path = OUTPUT_DIR / f"effnet_fold{fold}.weights.h5"
    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=6),
        # Save the best epoch, so predictions don't require retraining
        ModelCheckpoint(weights_path, monitor="val_accuracy", save_best_only=True, save_weights_only=True),
    ]
    finetune_history = model.fit(
        X_tr, y_tr, validation_data=(X_val, y_val), epochs=FINETUNE_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks,
    )
    model.load_weights(weights_path)

    history = {key: head_history.history[key] + finetune_history.history[key] for key in finetune_history.history}
    return model, history

In [ ]:
oof_probs = np.zeros((len(X_train), num_classes))  # out-of-fold predictions: each image predicted by the model that did not train on it
test_probs = {}
histories = {}

for fold in FOLDS_TO_RUN:
    print(f"===== Fold {fold + 1}/{N_FOLDS} =====")
    model, histories[fold] = train_fold(fold)

    is_val = fold_of == fold
    oof_probs[is_val] = model.predict(X_train[is_val])
    test_probs[fold] = model.predict(X_test)
    np.save(OUTPUT_DIR / f"test_probs_fold{fold}.npy", test_probs[fold])

    fold_acc = np.mean(oof_probs[is_val].argmax(axis=1) == y_train_int[is_val])
    print(f"Fold {fold + 1} validation accuracy: {fold_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, len(FOLDS_TO_RUN), figsize=(4 * len(FOLDS_TO_RUN), 3.5), sharey=True, squeeze=False)
for ax, fold in zip(axes[0], FOLDS_TO_RUN):
    pd.DataFrame(histories[fold])[["accuracy", "val_accuracy"]].plot(ax=ax, grid=True)
    ax.axvline(HEAD_EPOCHS - 0.5, color="gray", linestyle="--")  # start of fine-tuning
    ax.set_title(f"Fold {fold + 1}")
    ax.set_xlabel("Epoch")
plt.show()

### Evaluation

The cross-validation accuracy is the mean of the 5 fold accuracies; the spread across folds shows how much the score depends on the split. The confusion matrix uses the out-of-fold predictions, i.e. all 9,879 training images, each predicted by the model that did not train on it.

In [ ]:
fold_accs = pd.Series(
    {fold + 1: np.mean(oof_probs[fold_of == fold].argmax(axis=1) == y_train_int[fold_of == fold]) for fold in FOLDS_TO_RUN},
    name="validation accuracy",
).rename_axis("fold")
display(fold_accs.to_frame())
print(f"Cross-validation accuracy: {fold_accs.mean():.4f} ± {fold_accs.std():.4f}")

In [ ]:
# Per-vendor results on the out-of-fold predictions: rows are true vendors, columns are predictions
was_validated = np.isin(fold_of, FOLDS_TO_RUN)
val_true = y_train[was_validated]
val_pred = classes[oof_probs[was_validated].argmax(axis=1)]

confusion = pd.crosstab(val_true, val_pred, rownames=["true"], colnames=["predicted"])
confusion = confusion.reindex(index=classes, columns=classes, fill_value=0)
# Recall: share of each vendor's images that were predicted correctly
recall = pd.Series(val_true == val_pred).groupby(val_true).mean().rename("recall")
display(confusion)
display(recall.sort_values().to_frame())

## Submission

The test predictions are the average of the fold models' predicted probabilities (ensemble).

In [ ]:
y_pred = classes[np.mean(list(test_probs.values()), axis=0).argmax(axis=1)]

submission = pd.DataFrame({
    "Id": X_test_ids,
    "Label": y_pred
})
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
submission.head()